In [2]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR, SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
import pickle

# Load
df = pd.read_csv(r"D:\Summer_training\movieRecommendSVM\BollywoodMovieDetail.csv")
df.dropna(subset=['genre','actors','directors','hitFlop'], inplace=True)
df.drop_duplicates(inplace=True)
df['sequel'] = df['sequel'].fillna(0)
df['releaseYear'] = df['releaseYear'].fillna(0)
df.reset_index(drop=True, inplace=True)

# Features
df['primary_genre'] = df['genre'].apply(lambda x: str(x).split('|')[0].strip())
df['primary_actor'] = df['actors'].apply(lambda x: str(x).split('|')[0].strip())
df['primary_director'] = df['directors'].apply(lambda x: str(x).split('|')[0].strip())
df['hitFlop_label'] = df['hitFlop'].apply(lambda x: 'Hit' if x >= 6 else 'Average' if x >= 3 else 'Flop')

# Encode
le_genre = LabelEncoder()
le_actor = LabelEncoder()
le_director = LabelEncoder()
df['genre_enc'] = le_genre.fit_transform(df['primary_genre'])
df['actor_enc'] = le_actor.fit_transform(df['primary_actor'])
df['director_enc'] = le_director.fit_transform(df['primary_director'])

# Verify 3 Idiots
print(df[df['title']=='3 Idiots'][['title','hitFlop','hitFlop_label']])

# Train
X = df[['genre_enc','actor_enc','director_enc','releaseYear','sequel']]
y_reg = df['hitFlop'].astype(float)
y_clf = df['hitFlop_label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

svr = SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale')
svr.fit(X_scaled, y_reg)

svc = SVC(kernel='rbf', C=100, gamma='scale', random_state=42)
svc.fit(X_scaled, y_clf)

print("SVC Accuracy:", accuracy_score(y_clf, svc.predict(X_scaled)))

# Save
pickle.dump(svr, open('svr_model.pkl', 'wb'))
pickle.dump(svc, open('svc_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler_movie.pkl', 'wb'))
pickle.dump(le_genre, open('le_genre.pkl', 'wb'))
pickle.dump(le_actor, open('le_actor.pkl', 'wb'))
pickle.dump(le_director, open('le_director.pkl', 'wb'))
df.to_csv(r"D:\Summer_training\movieRecommendSVM\BollywoodMovieDetail.csv", index=False)

# Verify prediction
idx = df[df['title']=='3 Idiots'].index[0]
test = X_scaled[idx].reshape(1,-1)
print("3 Idiots SVC prediction:", svc.predict(test))
print("Sab save ho gaya!")

        title  hitFlop hitFlop_label
620  3 Idiots        9           Hit
SVC Accuracy: 0.7984313725490196
3 Idiots SVC prediction: ['Flop']
Sab save ho gaya!


In [24]:
df.to_csv(r"D:\Summer_training\movieRecommendSVM\bollywood_clean.csv", index=False)
print(df.columns.tolist())

['imdbId', 'title', 'releaseYear', 'releaseDate', 'genre', 'writers', 'actors', 'directors', 'sequel', 'hitFlop', 'primary_genre', 'primary_actor', 'primary_director', 'hitFlop_label', 'genre_enc', 'actor_enc', 'director_enc']


In [25]:
print(df[df['title'] == '3 Idiots'][['title', 'hitFlop', 'hitFlop_label']])

        title  hitFlop hitFlop_label
620  3 Idiots        9           Hit
